All Imports

In [ ]:
# All imports grouped at the top (required by assignment instructions)
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from IPython.display import display
import random
import warnings
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split

1.1 Load & show first 10 rows

In [114]:

# 1.1 Load Faceplate.csv and display first 10 transactions
df_face = pd.read_csv('Faceplate.csv')

# Remove Transaction column (it is just row numbering)
if 'Transaction' in df_face.columns:
    df_face = df_face.drop(columns=['Transaction'])

print("1.1 First 10 transactions:")
display(df_face.head(10))


1.1 First 10 transactions:


,Red,White,Blue,Orange,Green,Yellow
0,1,1,0,0,1,0
1,0,1,0,1,0,0
2,0,1,1,0,0,0
3,1,1,0,1,0,0
4,1,0,1,0,0,0
5,0,1,1,0,0,0
6,1,0,1,0,0,0
7,1,1,1,0,1,0
8,1,1,1,0,0,0
9,0,0,0,0,0,1


1.2 Support of {red, white}

In [115]:
# 1.2 Support of the itemset {red, white}
support_red_white = (df_face['Red'] & df_face['White']).mean()

print("1.2 Support of {red, white}:")
print(f"  → {support_red_white:.3f}  ({int(support_red_white * len(df_face))} out of {len(df_face)} transactions)")

1.2 Support of {red, white}:
  → 0.400  (4 out of 10 transactions)


2.Apriori Algorithm Application (20 points)

2.1 Frequent itemsets (min support 0.2)

In [116]:

# 2.1 Frequent itemsets with min_support = 0.2
frequent_itemsets = apriori(df_face, min_support=0.2, use_colnames=True)

frequent_itemsets = frequent_itemsets.sort_values('support', ascending=False)

print("2.1 Frequent itemsets with their support values:")
display(frequent_itemsets[['support', 'itemsets']])

2.1 Frequent itemsets with their support values:


,support,itemsets
1,0.7,frozenset({White})
0,0.6,frozenset({Red})
2,0.6,frozenset({Blue})
6,0.4,"frozenset({Blue, Red})"
5,0.4,"frozenset({White, Red})"
8,0.4,"frozenset({White, Blue})"
3,0.2,frozenset({Orange})
4,0.2,frozenset({Green})
7,0.2,"frozenset({Green, Red})"
9,0.2,"frozenset({Orange, White})"


2.2 Generate rules & sort by lift

In [117]:
# 2.2 Generate association rules (min confidence 0.5) and sort by lift descending
rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5)

rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print("2.2 Association rules sorted by lift ratio (descending):")
display(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']])

2.2 Association rules sorted by lift ratio (descending):


,antecedents,consequents,support,confidence,lift
0,frozenset({Green}),"frozenset({White, Red})",0.2,1.000000,2.500000
1,"frozenset({White, Red})",frozenset({Green}),0.2,0.500000,2.500000
2,frozenset({Green}),frozenset({Red}),0.2,1.000000,1.666667
3,"frozenset({Green, White})",frozenset({Red}),0.2,1.000000,1.666667
4,frozenset({Green}),frozenset({White}),0.2,1.000000,1.428571
5,frozenset({Orange}),frozenset({White}),0.2,1.000000,1.428571
6,"frozenset({Green, Red})",frozenset({White}),0.2,1.000000,1.428571
7,frozenset({Blue}),frozenset({Red}),0.4,0.666667,1.111111
8,frozenset({Red}),frozenset({Blue}),0.4,0.666667,1.111111
9,frozenset({Blue}),frozenset({White}),0.4,0.666667,0.952381


2.3 Top 6 rules (only requested columns)

In [118]:
# 2.3 Top 6 rules by lift – only the columns asked for
top6 = rules.head(6)[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']]

print("2.3 Top 6 rules by lift ratio:")
display(top6)


2.3 Top 6 rules by lift ratio:


,antecedents,consequents,support,confidence,lift,leverage
0,frozenset({Green}),"frozenset({White, Red})",0.2,1.0,2.500000,0.12
1,"frozenset({White, Red})",frozenset({Green}),0.2,0.5,2.500000,0.12
2,frozenset({Green}),frozenset({Red}),0.2,1.0,1.666667,0.08
3,"frozenset({Green, White})",frozenset({Red}),0.2,1.0,1.666667,0.08
4,frozenset({Green}),frozenset({White}),0.2,1.0,1.428571,0.06
5,frozenset({Orange}),frozenset({White}),0.2,1.0,1.428571,0.06


2.4 Interpretation of highest lift rule

In [119]:
# 2.4 Translate the rule with highest lift into a sentence
if len(rules) > 0:
    best_rule = rules.iloc[0]
    ante = list(best_rule['antecedents'])
    cons = list(best_rule['consequents'])
    conf = best_rule['confidence'] * 100
    lift = round(best_rule['lift'], 2)

    print("2.4 Rule with highest lift:")
    print(f"If {ante} are purchased, then with confidence {conf:.0f}% {cons} will also be purchased.")
    print(f"This rule has a lift ratio of {lift}.")
else:
    print("2.4 No association rules generated.")

2.4 Rule with highest lift:
If ['Green'] are purchased, then with confidence 100% ['White', 'Red'] will also be purchased.
This rule has a lift ratio of 2.5.


3.Book Purchase Association Rules (15 points)

3.1 Load CharlesBookClub & create binary matrix

In [120]:
# 3.1 Load CharlesBookClub.csv and create binary incidence matrix
df_book = pd.read_csv('CharlesBookClub.csv')

book_columns = [
    'ChildBks', 'YouthBks', 'CookBks', 'DoItYBks', 'RefBks',
    'ArtBks', 'GeogBks', 'ItalCook', 'ItalAtlas', 'ItalArt', 'Florence'
]

df_binary = df_book[book_columns].gt(0).astype(int)

print("3.1 First 10 rows of binary incidence matrix:")
display(df_binary.head(10))

3.1 First 10 rows of binary incidence matrix:


,ChildBks,YouthBks,CookBks,DoItYBks,RefBks,ArtBks,GeogBks,ItalCook,ItalAtlas,ItalArt,Florence
0,0,1,1,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,0
2,1,1,1,0,1,0,1,1,0,0,0
3,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,0,0,0,0,0,0,0
6,0,0,0,0,0,0,1,0,0,0,0
7,1,0,0,0,0,0,0,0,0,0,0
8,0,0,0,0,0,0,0,0,0,0,0
9,0,0,1,0,0,0,0,0,0,0,0


3.2 Frequent itemsets (min support 0.05)

In [121]:
# 3.2 Apriori with min_support = 0.05 (200 / 4000 transactions)
frequent_book = apriori(df_binary, min_support=0.05, use_colnames=True)

print(f"3.2 Number of frequent itemsets found: {len(frequent_book)}")


3.2 Number of frequent itemsets found: 61


3.3 Top 25 rules by lift

In [122]:
# 3.3 Generate rules (min confidence 0.5) and show top 25 by lift
rules_book = association_rules(frequent_book, metric="confidence", min_threshold=0.5)

rules_book = rules_book.sort_values('lift', ascending=False).reset_index(drop=True)

print("3.3 Top 25 rules sorted by lift ratio:")
display(rules_book.head(25)[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'leverage']])

3.3 Top 25 rules sorted by lift ratio:


,antecedents,consequents,support,confidence,lift,leverage
0,"frozenset({YouthBks, RefBks})","frozenset({ChildBks, CookBks})",0.05525,0.680000,2.809917,0.035588
1,"frozenset({RefBks, DoItYBks})","frozenset({ChildBks, CookBks})",0.06125,0.662162,2.736207,0.038865
2,"frozenset({YouthBks, DoItYBks})","frozenset({ChildBks, CookBks})",0.06700,0.648910,2.681448,0.042014
3,"frozenset({GeogBks, RefBks})","frozenset({ChildBks, CookBks})",0.05025,0.614679,2.539995,0.030467
4,"frozenset({YouthBks, GeogBks})","frozenset({ChildBks, CookBks})",0.06325,0.605263,2.501087,0.037961
5,"frozenset({GeogBks, DoItYBks})","frozenset({ChildBks, CookBks})",0.06050,0.599010,2.475248,0.036058
6,"frozenset({ChildBks, GeogBks, CookBks})",frozenset({YouthBks}),0.06325,0.577626,2.424452,0.037162
7,"frozenset({ChildBks, CookBks, RefBks})",frozenset({DoItYBks}),0.06125,0.591787,2.323013,0.034883
8,"frozenset({GeogBks, DoItYBks})",frozenset({YouthBks}),0.05450,0.539604,2.264864,0.030437
9,"frozenset({ChildBks, CookBks, RefBks})",frozenset({YouthBks}),0.05525,0.533816,2.240573,0.030591


4.Interpreting Association Rules Results (21 points)

4.1 Rule with highest support

In [123]:
# 4.1 Rule with highest support
if len(rules_book) > 0:
    highest_support = rules_book.loc[rules_book['support'].idxmax()]
    
    print("4.1 Rule with highest support:")
    print(f"Antecedents: {list(highest_support['antecedents'])}")
    print(f"Consequents: {list(highest_support['consequents'])}")
    print(f"Support    : {highest_support['support']:.4f}")
    print(f"Confidence : {highest_support['confidence']:.4f}")
    print(f"Lift       : {highest_support['lift']:.4f}")

4.1 Rule with highest support:
Antecedents: ['ChildBks']
Consequents: ['CookBks']
Support    : 0.2420
Confidence : 0.6142
Lift       : 1.4783


4.2 Highest lift + tradeoff discussion

In [124]:
# 4.2 Rule with highest lift + tradeoff discussion
if len(rules_book) > 0:
    highest_lift = rules_book.iloc[0]
    
    print("4.2 Rule with highest lift:")
    print(f"Antecedents: {list(highest_lift['antecedents'])}")
    print(f"Consequents: {list(highest_lift['consequents'])}")
    print(f"Support    : {highest_lift['support']:.4f}")
    
    print("\nTrade-off discussion:")
    print("  - High-support rules appear in many transactions → suitable for broad promotions")
    print("  - High-lift rules are very efficient but rare → better for targeted marketing")
    print("  - There is usually a trade-off between reach (support) and strength (lift)")


4.2 Rule with highest lift:
Antecedents: ['YouthBks', 'RefBks']
Consequents: ['ChildBks', 'CookBks']
Support    : 0.0553

Trade-off discussion:
  - High-support rules appear in many transactions → suitable for broad promotions
  - High-lift rules are very efficient but rare → better for targeted marketing
  - There is usually a trade-off between reach (support) and strength (lift)


4.3 Lowest confidence among top-10 lift

In [125]:

# 4.3 Lowest confidence among top 10 by lift
if len(rules_book) >= 10:
    top10_lift = rules_book.head(10)
    lowest_conf = top10_lift.loc[top10_lift['confidence'].idxmin()]
    
    print("4.3 Rule with lowest confidence among top-10 lift rules:")
    print(f"Antecedents : {list(lowest_conf['antecedents'])}")
    print(f"Consequents : {list(lowest_conf['consequents'])}")
    print(f"Confidence  : {lowest_conf['confidence']:.4f}")
    print(f"Lift        : {lowest_conf['lift']:.4f}")


4.3 Rule with lowest confidence among top-10 lift rules:
Antecedents : ['ChildBks', 'CookBks', 'RefBks']
Consequents : ['YouthBks']
Confidence  : 0.5338
Lift        : 2.2406


5.Association rules and chance effects (15 points)

5.1 Synthetic dataset

In [139]:

# 5.1 Create synthetic binary dataset (50 transactions, 9 items)
np.random.seed(0)
synthetic_data = np.random.randint(0, 2, size=(50, 9))
df_synthetic = pd.DataFrame(synthetic_data, columns=[f'Item_{i+1}' for i in range(9)])

print("5.1 Synthetic dataset (first 5 rows):")
display(df_synthetic.head())

5.1 Synthetic dataset (first 5 rows):


,Item_1,Item_2,Item_3,Item_4,Item_5,Item_6,Item_7,Item_8,Item_9
0,0,1,1,0,1,1,1,1,1
1,1,1,0,0,1,0,0,0,0
2,0,1,0,1,1,0,0,1,1
3,1,1,0,1,0,1,0,1,1
4,0,1,1,0,0,1,0,1,1


5.2 Apriori + rules on synthetic data

In [135]:

# 5.2 Apriori (min support 0.04) → rules (min confidence 0.7)
freq_synth = apriori(df_synthetic, min_support=0.04, use_colnames=True)
rules_synth = association_rules(freq_synth, metric="confidence", min_threshold=0.7)

print(f"5.2 Number of rules with confidence >= 0.7: {len(rules_synth)}")

5.2 Number of rules with confidence >= 0.7: 377


5.3 Top 6 rules by lift

In [136]:

# 5.3 Top 6 rules by lift (uplift)
if len(rules_synth) > 0:
    top6_synth = rules_synth.sort_values('lift', ascending=False).head(6)
    
    print("5.3 Top 6 rules by lift ratio:")
    display(top6_synth[['antecedents', 'consequents', 'support', 'confidence', 'lift']])
    
    print("\nObservation: Even in completely random data, some rules show high lift values.")
    print("This illustrates that high-lift rules can appear purely by chance.")

5.3 Top 6 rules by lift ratio:


,antecedents,consequents,support,confidence,lift
376,"frozenset({Item_9, Item_7, Item_8, Item_6})","frozenset({Item_3, Item_5, Item_2})",0.04,1.0,5.555556
356,"frozenset({Item_3, Item_5, Item_4, Item_6, Ite...","frozenset({Item_8, Item_1})",0.04,1.0,5.000000
370,"frozenset({Item_3, Item_5, Item_6, Item_9, Ite...","frozenset({Item_8, Item_7})",0.04,1.0,5.000000
327,"frozenset({Item_4, Item_5, Item_2, Item_7})","frozenset({Item_8, Item_6})",0.04,1.0,4.545455
347,"frozenset({Item_3, Item_6, Item_8, Item_1, Ite...","frozenset({Item_4, Item_7})",0.04,1.0,4.545455
282,"frozenset({Item_8, Item_3, Item_1, Item_6})","frozenset({Item_4, Item_7})",0.06,1.0,4.545455



Observation: Even in completely random data, some rules show high lift values.
This illustrates that high-lift rules can appear purely by chance.


6.Item-Based Collaborative Filtering (20 points)

6.1 Synthetic ratings

In [137]:

# 6.1 Create synthetic ratings dataset (random seed = 0)
random.seed(0)

ratings_list = []
for _ in range(5000):
    user = random.randint(0, 999)
    item = random.randint(0, 99)
    rating = random.randint(1, 5)
    ratings_list.append([user, item, rating])

df_ratings = pd.DataFrame(ratings_list, columns=['userID', 'itemID', 'rating'])

print("6.1 First 10 rows of synthetic ratings dataset:")
display(df_ratings.head(10))

6.1 First 10 rows of synthetic ratings dataset:


,userID,itemID,rating
0,864,49,4
1,41,33,5
2,497,51,3
3,991,61,3
4,597,27,5
5,142,36,2
6,773,12,5
7,818,32,5
8,722,77,2
9,317,12,1


6.2 Train/test split + matrix dimensions

In [130]:

# 6.2 Convert to surprise format + train/test split
#(this cell requires scikit-surprise – skip execution locally if needed)

try:
    from surprise import Dataset, Reader
    from surprise.model_selection import train_test_split

    reader = Reader(rating_scale=(1, 5))
    data = Dataset.load_from_df(df_ratings[['userID', 'itemID', 'rating']], reader)

    trainset, testset = train_test_split(data, test_size=0.2, random_state=0)

    print("6.2 Dimensions of training and test sets:")
    print(f"  Training set: {trainset.n_ratings} ratings")
    print(f"  Test set:     {len(testset)} ratings")
except ModuleNotFoundError:
    print("surprise package not available locally – this will run on Gradescope")

surprise package not available locally – this will run on Gradescope


6.3 Item-based cosine similarity

In [131]:

# 6.3 Build item-based collaborative filtering model with cosine similarity

try:
    from surprise import KNNBasic

    sim_options = {
        'name': 'cosine',
        'user_based': False  # item-based
    }

    algo = KNNBasic(sim_options=sim_options)
    algo.fit(trainset)

    print("6.3 Item-based KNN model with cosine similarity trained successfully.")
except NameError:
    print("trainset not defined locally – skip this cell")
except ModuleNotFoundError:
    print("surprise package not available locally – this will run on Gradescope")

surprise package not available locally – this will run on Gradescope


6.4 Recommendations (first 5 users)


In [ ]:

# 6.4 Predict ratings for unseen pairs and show top recommendations
# (showing first 10 users)

try:
    from collections import defaultdict

    anti_testset = trainset.build_anti_testset()
    predictions = algo.test(anti_testset)

    user_preds = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        user_preds[uid].append((iid, est))

    print("6.4 Top 5 recommended items for first 10 users:")
    for uid in sorted(user_preds)[:10]:
        top5 = sorted(user_preds[uid], key=lambda x: x[1], reverse=True)[:5]
        items = [iid for iid, _ in top5]
        scores = [round(score, 2) for _, score in top5]
        print(f"User {uid}: items = {items}, predicted ratings = {scores}")
except NameError:
    print("Cannot run predictions locally (trainset/algo not defined)")
except ModuleNotFoundError:
    print("surprise package not available locally – this will run on Gradescope")


Cannot run predictions locally (trainset/algo not defined)
